# END_DIP — Traffic Sign + Car License Plate Detection
Run top-to-bottom in Google Colab. Enable a GPU first: **Runtime → Change runtime type → T4 GPU**.

The full-resolution result is saved to Google Drive. A smaller H.264 preview is created in `/content` so it plays reliably inside Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/DIP
!git clone -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import os, torch
VIDEO = '/content/drive/MyDrive/DIP/video1.mp4'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Video exists:', os.path.exists(VIDEO), VIDEO)
assert os.path.exists(VIDEO), 'Put video1.mp4 in MyDrive/DIP first.'


## Download models once
Models are cached under `MyDrive/DIP/models/`, so a Colab restart does not require re-downloading them.

In [ ]:
!python download_models.py --models-dir '/content/drive/MyDrive/DIP/models'


## Process `video1.mp4`
Only traffic-sign boxes and car/bus/truck license-plate boxes are drawn. Vehicle boxes are used internally and stay hidden.

In [ ]:
!python main.py \
  --input '/content/drive/MyDrive/DIP/video1.mp4' \
  --output-dir '/content/drive/MyDrive/DIP/outputs' \
  --models-dir '/content/drive/MyDrive/DIP/models' \
  --sign-conf 0.25 \
  --plate-conf 0.30 \
  --vehicle-conf 0.30


## Preview result directly in Colab
The Drive file is kept at full quality. This cell creates a local browser-compatible H.264 preview.

In [ ]:
import os, subprocess
from IPython.display import Video, display

DRIVE_OUT = '/content/drive/MyDrive/DIP/outputs/video1_result.mp4'
PREVIEW = '/content/video1_result_preview.mp4'
assert os.path.exists(DRIVE_OUT), f'Output not found: {DRIVE_OUT}'

cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', DRIVE_OUT,
    '-vf', 'scale=960:-2',
    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '27',
    '-pix_fmt', 'yuv420p', '-tag:v', 'avc1',
    '-movflags', '+faststart', '-an', PREVIEW,
]
subprocess.run(cmd, check=True)
print(f'Full result: {DRIVE_OUT} ({os.path.getsize(DRIVE_OUT)/1024/1024:.1f} MB)')
print(f'Preview: {PREVIEW} ({os.path.getsize(PREVIEW)/1024/1024:.1f} MB)')
display(Video(PREVIEW, embed=True, width=900, html_attributes='controls'))


## Outputs
- `MyDrive/DIP/outputs/video1_result.mp4` — annotated video
- `MyDrive/DIP/outputs/video1_result.csv` — detections
- `MyDrive/DIP/outputs/video1_plates/` — best plate crops per tracked car/bus/truck

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/DIP/outputs'))


## Optional: Gradio UI for another video
Upload another video and tune the sign/plate confidence without changing code.

In [ ]:
!python app.py


## Optional: fine-tune the plate detector on Vietnamese car plates
The default pretrained plate detector runs immediately. Fine-tuning is optional and automatically creates `MyDrive/DIP/models/plate_best.pt`, which the pipeline will prefer on later runs.

In [ ]:
# import os
# os.environ['ROBOFLOW_API_KEY'] = 'YOUR_KEY'
# !python training/prepare_plate_dataset.py --target '/content/drive/MyDrive/DIP/datasets/vn_car_plate'
# !python training/train_plate.py --data '/content/drive/MyDrive/DIP/datasets/vn_car_plate/data.yaml' --models-dir '/content/drive/MyDrive/DIP/models' --runs-dir '/content/drive/MyDrive/DIP/training_runs' --epochs 20 --batch 16
